In [4]:
from Chatbot import (
    retrieve_context,
    save_chat_history,
    fetch_chat_history,
    get_gemini_response,
)

In [5]:
import os
import numpy as np
import pandas as pd
import json
import time
import re
import random
from datetime import datetime
from typing import List, Dict, Any, Generator
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pinecone import Pinecone, ServerlessSpec
import fitz  # PyMuPDF
import docx2txt  # Word extraction
import serpapi
import langgraph
import langchain
import serpapi
import wikipedia
from groq import Groq
from langchain_groq import ChatGroq
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI
from google import genai

In [6]:
# ---------------- LOAD ENV VARIABLES ----------------
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME", "studybuddy")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT", "us-east-1")
DEFAULT_TIMEZONE = os.getenv("DEFAULT_TIMEZONE", "UTC")

if not GEMINI_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY not found!")
if not PINECONE_API_KEY:
    raise ValueError("❌ PINECONE_API_KEY not found!")

In [7]:
# test_regex.py
from Chatbot import retrieve_context

query = "Given the regular expression [A-Z][a-z]* [ ][A-Z][A-Z] from the text, what pattern in text does it represent?"

context = retrieve_context(query)
print(f"Context found: {bool(context)}")
print(f"Preview: {context[:500] if context else 'No context'}")

  [DEBUG] Found 5 matches for: Given the regular expression [...
  [DEBUG] Score: 0.755 - Source: notes - Text length: 695
  [DEBUG] Score: 0.724 - Source: notes - Text length: 793
  [DEBUG] Score: 0.703 - Source: notes - Text length: 211
  [DEBUG] Score: 0.664 - Source: notes - Text length: 762
  [DEBUG] Score: 0.645 - Source: notes - Text length: 749
  [DEBUG] Returning 3 contexts, total length: 1752
Context found: True
Preview: [From notes] That is, L(EF) = L(E)L(F). 3. If E is a regular expression, then E∗is a regular expression denoting the closure
of L(E). That is, L(E∗) = (L(E))∗. 4. If E is a regular expression, then (E) is a regular expression denoting the same
as E. Formally, L((E)) = (L(E)). Automata Theory, Languages and Computation - M´ırian Halfeld-Ferrari – p. 17/1

The use of regular expressions: examples of
applications
Deﬁnition of lexical analysers (compilers). Used in operation systems like UNIX (a Uni


In [8]:
eval_questions = []
with open('generated_questions.txt', 'r', encoding='utf-8') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [9]:
# Get answers for all questions
answers = []
for question in eval_questions:
    try:
        # Get response (handles both streaming and non-streaming)
        response = get_gemini_response(question, user_id="eval_user")
        
        # If it's a generator (streaming), collect all chunks
        if hasattr(response, '__iter__') and not isinstance(response, str):
            full_response = ""
            for chunk in response:
                if chunk:
                    full_response += chunk
            answers.append(full_response)
        else:
            answers.append(str(response))
            
        print(f"✅ Processed: {question[:50]}...")
        
    except Exception as e:
        print(f"❌ Error: {question[:50]}... - {e}")
        answers.append(f"ERROR: {str(e)}")

# Print results
print(f"\n✅ Completed {len(answers)}/{len(eval_questions)} questions")

✅ Processed: What is an alphabet in the context of automata the...
✅ Processed: What is the symbol used to denote the empty string...
✅ Processed: What is the Kleene star operator (Σ*) used for?...
✅ Processed: What is a formal language?...
✅ Processed: According to the text, what is the difference betw...

🔧 Using tools:
   - web_search: software applications using finite automata
✅ Processed: List at least three different software application...
✅ Processed: Explain the difference between concatenating two s...
✅ Processed: Why is the empty string (ε) considered the identit...
✅ Processed: A language L is defined as {0ⁿ1ⁿ | n ≥ 1}. Write o...
[WARNING] Agent streaming failed: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': "To analyze the given regular expression and its pattern, as well as discuss its limitations 

In [ ]:
 #pip install jsonschema torch torchvision jsonschema torchaudio --index-url https://download.pytorch.org/whl/cpu 

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# trulens_working.py
import os
import numpy as np
import torch
from groq import Groq
from trulens.core import TruSession, Metric
from Chatbot import retrieve_context

# ============================================
# 1. PATCH TORCH (for Python 3.13)
# ============================================

if not hasattr(torch, 'Tensor'):
    class Tensor:
        pass
    torch.Tensor = Tensor
    print("✅ Patched torch.Tensor")

# ============================================
# 2. YOUR DATA
# ============================================

eval_questions = [
    "What is an alphabet in automata theory?",
    "What is the symbol used to denote the empty string?",
    "What is the Kleene star operator (Σ*) used for?",
    "What is a formal language?",
    "Difference between ∅ and {ε}?"
]

answers = [
    "In automata theory, an alphabet is a finite set of symbols...",
    "The symbol used to denote the empty string is ε (epsilon).",
    "The Kleene star operator represents the set of all strings...",
    "A formal language is a set of strings of symbols...",
    "The empty language (∅) has no strings, while {ε} has one string..."
]

# ============================================
# 3. SET API KEY
# ============================================


# ============================================
# 4. RETRIEVE CONTEXTS
# ============================================

print("🔄 Retrieving contexts...")
contexts = []
for q in eval_questions:
    context = retrieve_context(q, user_id="default_user_id", top_k=3)
    contexts.append([context])
print("✅ Contexts retrieved")

# ============================================
# 5. CREATE CUSTOM METRICS
# ============================================

client = Groq(api_key=GROQ_API_KEY)

def evaluate_context_relevance(input: str, context: list) -> float:
    """Evaluate if context is relevant to the question"""
    prompt = f"""Rate the relevance of the context to the question on a scale of 0-1.
    Return ONLY a number between 0 and 1.
    
    Question: {input}
    Context: {context[0][:500] if context else "No context"}
    
    Relevance score (0-1):"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 0.5

def evaluate_answer_relevance(input: str, output: str) -> float:
    """Evaluate if the answer is relevant to the question"""
    prompt = f"""Rate the relevance of the answer to the question on a scale of 0-1.
    Return ONLY a number between 0 and 1.
    
    Question: {input}
    Answer: {output}
    
    Relevance score (0-1):"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 0.5

def evaluate_groundedness(context: list, output: str) -> float:
    """Evaluate if the answer is grounded in the context"""
    prompt = f"""Rate how grounded the answer is in the context on a scale of 0-1.
    Return ONLY a number between 0 and 1.
    
    Context: {context[0][:500] if context else "No context"}
    Answer: {output}
    
    Groundedness score (0-1):"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 0.5

print("✅ Custom metrics created!")

# ============================================
# 6. SETUP TRULENS SESSION
# ============================================

session = TruSession()

# Create Metrics
f_context_relevance = (
    Metric(evaluate_context_relevance)
    .on_input()
    .on_context(collect_list=False)
    .aggregate(np.mean)
)

f_answer_relevance = (
    Metric(evaluate_answer_relevance)
    .on_input()
    .on_output()
)

f_groundedness = (
    Metric(evaluate_groundedness)
    .on_context(collect_list=False)
    .on_output()
)

print("✅ Metrics defined!")

# ============================================
# 7. RUN EVALUATION
# ============================================

print("\n🚀 Running Evaluation with Metrics...")

# Create records manually
records = []
for i, (q, a, c) in enumerate(zip(eval_questions, answers, contexts), 1):
    print(f"📝 Evaluating Q{i}: {q[:50]}...")
    
    # Calculate scores manually
    context_score = evaluate_context_relevance(q, c)
    answer_score = evaluate_answer_relevance(q, a)
    grounded_score = evaluate_groundedness(c, a)
    
    record = {
        "input": q,
        "output": a,
        "context": c,
        "context_relevance": context_score,
        "answer_relevance": answer_score,
        "groundedness": grounded_score
    }
    records.append(record)

# Convert to DataFrame
import pandas as pd
df = pd.DataFrame(records)

print("\n📊 Results:")
print(df[['input', 'context_relevance', 'answer_relevance', 'groundedness']].round(3))

# Averages
print("\n📈 Averages:")
print(f"Context Relevance: {df['context_relevance'].mean():.3f}")
print(f"Answer Relevance: {df['answer_relevance'].mean():.3f}")
print(f"Groundedness: {df['groundedness'].mean():.3f}")

# Save results
df.to_csv("trulens_results.csv", index=False)
print("\n💾 Results saved to trulens_results.csv")

# Try launching dashboard if available
try:
    print("\n🌐 Attempting to launch TruLens dashboard...")
    session.run_dashboard()
except Exception as e:
    print(f"Dashboard not available: {e}")
    print("Results are available in the DataFrame above.")

🔄 Retrieving contexts...
  [DEBUG] Found 0 matches for: What is an alphabet in automat...
  [DEBUG] No matches found at all!
  [DEBUG] Found 0 matches for: What is the symbol used to den...
  [DEBUG] No matches found at all!
  [DEBUG] Found 0 matches for: What is the Kleene star operat...
  [DEBUG] No matches found at all!
  [DEBUG] Found 0 matches for: What is a formal language?...
  [DEBUG] No matches found at all!
  [DEBUG] Found 0 matches for: Difference between ∅ and {ε}?...
  [DEBUG] No matches found at all!
✅ Contexts retrieved


Metric implementation <function evaluate_context_relevance at 0x000002378F1E3380> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Metric implementation <function evaluate_answer_relevance at 0x000002378F1E2D40> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Metric implementation <function evaluate_groundedness at 0x000002378F1E31A0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


✅ Custom metrics created!
✅ Metrics defined!

🚀 Running Evaluation with Metrics...
📝 Evaluating Q1: What is an alphabet in automata theory?...


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [ ]:
# diagnostic.py
import sys
import os
import subprocess

print("=" * 70)
print("🔍 ENVIRONMENT DIAGNOSTIC")
print("=" * 70)

# 1. Python version
print(f"\n🐍 Python Version: {sys.version}")
print(f"   Python Executable: {sys.executable}")

# 2. Installed packages
print("\n📦 Installed Packages:")
try:
    import pkg_resources
    packages = ['trulens', 'llama-index-core', 'pinecone', 'google-generativeai', 'groq']
    for pkg in packages:
        try:
            version = pkg_resources.get_distribution(pkg).version
            print(f"   {pkg}: {version}")
        except:
            print(f"   {pkg}: NOT INSTALLED")
except:
    # Try pip list instead
    result = subprocess.run(['pip', 'list'], capture_output=True, text=True)
    print(result.stdout)

# 3. Check if default.sqlite exists and its size
print("\n📁 Database Files:")
for file in ['default.sqlite', 'trulens_results.csv', 'trulens_records.csv']:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f"   {file}: {size} bytes")
    else:
        print(f"   {file}: NOT FOUND")

# 4. Check environment variables
print("\n🔧 Environment Variables:")
env_vars = ['TRULENS_OTEL_TRACING', 'TRULENS_OTEL_ENABLED', 'PYTHONPATH']
for var in env_vars:
    value = os.getenv(var)
    print(f"   {var}: {value if value else '(not set)'}")

# 5. Test basic TruLens import
print("\n📦 Testing TruLens imports:")
try:
    from trulens.core import TruSession, Metric, FeedbackMode
    print("   ✅ TruSession, Metric, FeedbackMode imported")
except ImportError as e:
    print(f"   ❌ Import error: {e}")

try:
    from trulens.apps.app import TruApp
    print("   ✅ TruApp imported")
except ImportError as e:
    print(f"   ❌ Import error: {e}")

try:
    from trulens.dashboard import run_dashboard
    print("   ✅ run_dashboard imported")
except ImportError as e:
    print(f"   ❌ Import error: {e}")

print("\n" + "=" * 70)
print("✅ DIAGNOSTIC COMPLETE")
print("=" * 70)

In [11]:
%pip install trulens trulens-apps-llamaindex trulens-providers-openai

Note: you may need to restart the kernel to use updated packages.
